# 레슨 10 — 통합 프로젝트: 자동화 리포트 만들기

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/10/[학생용] 레슨 10 — 통합 프로젝트: 자동화 리포트 만들기.ipynb)

이 노트북은 읽기와 따라하기용 강의 노트북이다. HTML 파싱, 상대 URL, 자료 목록, 품질 검증, 저장, 로그 요약을 하나의 작은 업무 자동화 리포트로 묶는 최종 프로젝트을 안전한 합성 fixture로 연습한다.

## 학습 목표

1. 여러 HTML fixture를 하나의 포털 구조로 해석한다.
2. 링크, 공지, 과정, 다운로드 자료를 각각 추출한다.
3. 중복과 상태 오류를 점검해 저장 전 품질을 확인한다.
4. CSV, JSON, SQLite 산출물을 함께 만든다.
5. 운영자가 읽을 수 있는 3문장 자동화 메모를 작성한다.

---

## 1. 수업 맥락과 안전 기준

마지막 레슨은 “한 페이지에서 값을 뽑았다”가 아니라 “업무 담당자가 바로 확인할 수 있는 리포트”까지 만든다. 합성 포털을 대상으로 하므로 외부 사이트 부하 없이 실제 운영 흐름을 통합 연습한다.

자동화는 빠르게 반복하는 도구이기 때문에 실패했을 때 더 위험해질 수 있다. 그래서 이번 레슨에서는 모든 입력을 수업용 파일로 고정하고, 결과를 저장하기 전에 검증하거나 로그를 남기는 과정을 코드에 포함한다. 이 습관은 실제 사이트를 대상으로 할 때 요청량을 줄이고, 오류를 빨리 발견하게 만든다.

## 2. 환경 셀


In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/10/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

def parse_html(filename):
    return BeautifulSoup(load_text(filename), 'html.parser')

def text_or_empty(el):
    return '' if el is None else el.get_text(' ', strip=True)

def status_ok(status):
    return str(status).strip().lower() in {'open', 'ready', 'published'}

def make_key(*parts):
    return '::'.join(str(part).strip().lower() for part in parts)



---

## 3. 핵심 개념

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
index = parse_html('portal_index.html')
print(text_or_empty(index.select_one('h1')))


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 4. 자료 구조 확인

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
links = []
for a in index.select('a[data-page]'):
    links.append({'label': a.get_text(' ', strip=True), 'href': a['href'], 'url': urljoin('https://lesson.local/portal/', a['href'])})
print(links)


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 5. 품질 기준 적용

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
notice = parse_html('portal_notice.html')
notices = [{'title': item.select_one('.title').get_text(' ', strip=True), 'level': item.get('data-level'), 'date': item.get('data-date')} for item in notice.select('.notice-card')]
print(notices[:2])


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 6. 저장과 보고

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
courses = parse_html('portal_courses.html')
course_rows = []
for row in courses.select('tbody tr'):
    cells = [td.get_text(' ', strip=True) for td in row.select('td')]
    course_rows.append({'course': cells[0], 'teacher': cells[1], 'students': clean_int(cells[2]), 'status': cells[3]})
print(course_rows[:2])


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 7. 운영 관점 점검

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
manifest = load_csv('download_manifest.csv')
print(manifest[:2])


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 8. 마무리 체크

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
rules = load_json('quality_rules.json')
ready_courses = [row for row in course_rows if status_ok(row['status']) and row['students'] >= rules['min_students_for_report']]
print(ready_courses)


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 9. 핵심 개념

아래 셀은 강사가 먼저 실행 흐름을 보여주고, 학생이 같은 구조를 자기 말로 설명하도록 만든 예제다. 코드는 짧지만 입력, 처리, 출력이 분리되어 있어 나중에 문제 풀이로 확장하기 쉽다.


In [ ]:
report = {'notice_count': len(notices), 'course_count': len(course_rows), 'file_count': len(manifest), 'ready_course_count': len(ready_courses)}
Path('lesson10_portal_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(report)


이 셀에서 확인해야 할 것은 출력값 하나가 아니라 데이터가 어떤 단계에서 바뀌었는지다. 자동화 수업에서는 중간 변수명을 명확히 두면 학생이 오류 위치를 쉽게 찾을 수 있다.

---

## 데이터 출처와 안전 규칙

portal_index.html은 합성 포털의 시작 페이지다. portal_notice.html, portal_courses.html, portal_downloads.html, portal_status.html은 각각 공지, 과정, 자료, 상태 정보를 담는다. download_manifest.csv와 quality_rules.json은 최종 리포트 검증에 사용한다.

- 모든 파일은 수업용 합성 데이터다.
- 실제 사이트에 반복 요청하지 않는다.
- 저장 파일은 레슨 폴더 또는 코랩 현재 작업 폴더에만 만든다.
- 외부 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 포함 여부를 먼저 확인한다.

### 보강 설명 1

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 2

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 3

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 4

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 5

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 6

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 7

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 8

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 9

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 10

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 11

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 12

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 13

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 14

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 15

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 16

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 17

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 18

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 19

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 20

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 21

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 22

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 23

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 24

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 25

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 26

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 27

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 28

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.

### 보강 설명 29

학생에게는 코드의 속도보다 검증 순서를 강조한다. 입력을 읽고, 구조를 확인하고, 오류를 분류하고, 저장한 뒤 요약을 남기는 순서가 유지되면 도구가 바뀌어도 자동화 품질은 크게 흔들리지 않는다.


# 레슨 10 — 실습 문제

통합 프로젝트: 자동화 리포트 만들기 레슨의 학생용 문제 노트북이다. 강의 노트북을 먼저 실행한 뒤 빈칸을 직접 채운다.

## 통과 기준

- 총 15문제 중 12문제 이상 정상 출력이면 통과.
- 문제 1~5는 구조 확인, 6~10은 응용 처리, 11~15는 저장과 운영 요약이다.
- 정답값은 적지 않는다. 출력 형태와 fixture 구조를 보고 직접 판단한다.

## 0. 환경 셀


In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/10/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

def parse_html(filename):
    return BeautifulSoup(load_text(filename), 'html.parser')

def text_or_empty(el):
    return '' if el is None else el.get_text(' ', strip=True)

def status_ok(status):
    return str(status).strip().lower() in {'open', 'ready', 'published'}

def make_key(*parts):
    return '::'.join(str(part).strip().lower() for part in parts)



---

## 문제 1 — 포털 제목 읽기

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 시작 HTML 파일과 제목 selector를 채운다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
index = parse_html('____')
print(text_or_empty(index.select_one('____')))



---

## 문제 2 — 목차 링크 수집

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 목차 링크에는 data-page 속성이 있다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
links = []
for a in index.select('____'):
    links.append({'label': a.get_text(' ', strip=True), 'href': a['href']})
print(links)



---

## 문제 3 — 상대 URL을 절대 URL로 바꾸기

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: href 값을 사용한다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
base = 'https://lesson.local/portal/'
absolute = [urljoin(base, item['____']) for item in links]
print(absolute)



---

## 문제 4 — 공지 카드 추출

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 공지 파일과 카드 selector를 찾는다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
notice = parse_html('____')
notice_cards = notice.select('____')
print(len(notice_cards))



---

## 문제 5 — 공지 제목과 레벨 정리

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 제목 class와 data 속성 이름을 채운다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
notices = []
for card in notice_cards:
    notices.append({'title': text_or_empty(card.select_one('____')), 'level': card.get('____'), 'date': card.get('data-date')})
print(notices[:3])



---

## 문제 6 — 과정 표 행 읽기

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 과정 HTML과 표 셀 selector를 넣는다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
courses = parse_html('____')
course_rows = []
for row in courses.select('tbody tr'):
    cells = [td.get_text(' ', strip=True) for td in row.select('____')]
    course_rows.append({'course': cells[0], 'teacher': cells[1], 'students': clean_int(cells[2]), 'status': cells[3]})
print(course_rows[:2])



---

## 문제 7 — 다운로드 manifest 읽기

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 자료 목록 CSV와 파일명 컬럼을 확인한다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
manifest = load_csv('____')
print(manifest[0]['____'])



---

## 문제 8 — 상태 페이지 metric 추출

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 상태 HTML과 metric selector를 채운다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
status_page = parse_html('____')
metrics = {item.get('data-name'): clean_int(item.get_text(' ', strip=True)) for item in status_page.select('____')}
print(metrics)



---

## 문제 9 — 품질 규칙 읽기

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 규칙 파일과 최소 학생 수 키를 찾는다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
rules = load_json('____')
print(rules['____'])



---

## 문제 10 — 리포트 대상 과정 필터링

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 최소 학생 수 키를 넣는다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
ready_courses = []
for row in course_rows:
    if status_ok(row['status']) and row['students'] >= rules['____']:
        ready_courses.append(row)
print(ready_courses)



---

## 문제 11 — 자료 파일 중복 키 만들기

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 파일명 컬럼을 넣는다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
keys = [make_key(row['course'], row['____']) for row in manifest]
print(len(keys), len(set(keys)))



---

## 문제 12 — 통합 CSV 행 만들기

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 과정별 자료 목록 변수를 넣는다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
report_rows = []
for row in ready_courses:
    files = [item for item in manifest if item['course'] == row['course']]
    report_rows.append({'course': row['course'], 'teacher': row['teacher'], 'students': row['students'], 'file_count': len(____), 'status': row['status']})
print(report_rows)



---

## 문제 13 — CSV와 JSON 저장

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 전체 자료 목록 개수를 넣는다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
write_csv('lesson10_portal_report.csv', report_rows)
summary = {'notice_count': len(notices), 'course_count': len(course_rows), 'ready_course_count': len(report_rows), 'file_count': len(____)}
Path('lesson10_portal_report.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(summary)



---

## 문제 14 — SQLite 저장과 조회

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 저장할 통합 행 목록을 넣는다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
conn = sqlite3.connect('lesson10_portal.db')
conn.execute('drop table if exists report')
conn.execute('create table report(course text, teacher text, students integer, file_count integer, status text)')
conn.executemany('insert into report values(:course, :teacher, :students, :file_count, :status)', ____)
print(conn.execute('select course, file_count from report order by students desc').fetchall())
conn.close()



---

## 문제 15 — 운영 메모 3문장 작성

지시된 값을 코드로 추출하거나 상태를 변경한다. 실행 결과는 한 줄 출력, 리스트, 딕셔너리, 저장 파일 생성 중 하나다.

**기대 결과 형태**: 요구한 값이 출력되거나 지정한 파일이 생성된다.

**빈칸 힌트**: 작성한 문장 리스트를 출력한다. 완성 코드를 그대로 따라 쓰지 말고 `____` 부분만 스스로 판단한다.


In [ ]:
memo = [
    f"공지 {len(notices)}건, 과정 {len(course_rows)}건을 확인했습니다.",
    f"리포트 대상 과정은 {len(report_rows)}건이며 자료 파일은 {len(manifest)}개입니다.",
    f"최근 실행 수는 {metrics.get('runs', 0)}회이고 오류 수는 {metrics.get('errors', 0)}회입니다.",
]
print('\n'.join(____))



### 보강 설명 1

문제가 막히면 HTML 태그, CSV 헤더, JSON 키를 먼저 소리 내어 읽는다. 함수명을 떠올리기 전에 입력 데이터의 구조를 확인하면 빈칸을 더 안정적으로 채울 수 있다.

### 보강 설명 2

문제가 막히면 HTML 태그, CSV 헤더, JSON 키를 먼저 소리 내어 읽는다. 함수명을 떠올리기 전에 입력 데이터의 구조를 확인하면 빈칸을 더 안정적으로 채울 수 있다.

### 보강 설명 3

문제가 막히면 HTML 태그, CSV 헤더, JSON 키를 먼저 소리 내어 읽는다. 함수명을 떠올리기 전에 입력 데이터의 구조를 확인하면 빈칸을 더 안정적으로 채울 수 있다.

### 보강 설명 4

문제가 막히면 HTML 태그, CSV 헤더, JSON 키를 먼저 소리 내어 읽는다. 함수명을 떠올리기 전에 입력 데이터의 구조를 확인하면 빈칸을 더 안정적으로 채울 수 있다.

### 보강 설명 5

문제가 막히면 HTML 태그, CSV 헤더, JSON 키를 먼저 소리 내어 읽는다. 함수명을 떠올리기 전에 입력 데이터의 구조를 확인하면 빈칸을 더 안정적으로 채울 수 있다.

### 보강 설명 6

문제가 막히면 HTML 태그, CSV 헤더, JSON 키를 먼저 소리 내어 읽는다. 함수명을 떠올리기 전에 입력 데이터의 구조를 확인하면 빈칸을 더 안정적으로 채울 수 있다.


# 레슨 10 — 최종 미션

## 프로젝트: 통합 프로젝트: 자동화 리포트 만들기 자동화 리포트

이번 레슨에서 배운 내용을 하나의 작은 운영 자동화로 묶는다. 제공된 fixture만 사용하고 외부 사이트에는 요청하지 않는다.

## 요구사항

1. 레슨의 주요 입력 파일을 모두 읽는다.
2. 최소 2개 이상의 검증 기준을 적용한다.
3. 중복, 오류, 성공 건수를 분리해서 계산한다.
4. 결과 CSV 또는 JSON 중 하나 이상을 저장한다.
5. 마지막에 운영자가 읽을 수 있는 3문장 요약을 작성한다.

## 보너스

- SQLite 저장 또는 상태 코드별 요약을 추가한다.
- 오류가 있는 항목만 따로 저장한다.

## 시작 코드


In [ ]:
import os
import re
import csv
import json
import time
import sqlite3
import logging
from pathlib import Path
from urllib.parse import urljoin, urlparse

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/10/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_csv(filename):
    text = load_text(filename)
    return list(csv.DictReader(text.splitlines()))

def load_json(filename):
    return json.loads(load_text(filename))

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)) or '0')

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

def parse_html(filename):
    return BeautifulSoup(load_text(filename), 'html.parser')

def text_or_empty(el):
    return '' if el is None else el.get_text(' ', strip=True)

def status_ok(status):
    return str(status).strip().lower() in {'open', 'ready', 'published'}

def make_key(*parts):
    return '::'.join(str(part).strip().lower() for part in parts)



## 자동화 결과 요약

- 오늘 확인한 입력:
- 저장한 산출물:
- 다음에 개선할 점:

### 보강 설명 1

최종 미션은 학생이 자기 노트북에서 실행 결과와 저장 파일을 함께 제출하는 형태로 운영한다. 강사는 결과 파일 이름, 행 수, 요약 문장 세 가지를 우선 확인하면 빠르게 피드백할 수 있다.

### 보강 설명 2

최종 미션은 학생이 자기 노트북에서 실행 결과와 저장 파일을 함께 제출하는 형태로 운영한다. 강사는 결과 파일 이름, 행 수, 요약 문장 세 가지를 우선 확인하면 빠르게 피드백할 수 있다.

### 보강 설명 3

최종 미션은 학생이 자기 노트북에서 실행 결과와 저장 파일을 함께 제출하는 형태로 운영한다. 강사는 결과 파일 이름, 행 수, 요약 문장 세 가지를 우선 확인하면 빠르게 피드백할 수 있다.

### 보강 설명 4

최종 미션은 학생이 자기 노트북에서 실행 결과와 저장 파일을 함께 제출하는 형태로 운영한다. 강사는 결과 파일 이름, 행 수, 요약 문장 세 가지를 우선 확인하면 빠르게 피드백할 수 있다.

### 보강 설명 5

최종 미션은 학생이 자기 노트북에서 실행 결과와 저장 파일을 함께 제출하는 형태로 운영한다. 강사는 결과 파일 이름, 행 수, 요약 문장 세 가지를 우선 확인하면 빠르게 피드백할 수 있다.

### 보강 설명 6

최종 미션은 학생이 자기 노트북에서 실행 결과와 저장 파일을 함께 제출하는 형태로 운영한다. 강사는 결과 파일 이름, 행 수, 요약 문장 세 가지를 우선 확인하면 빠르게 피드백할 수 있다.

### 보강 설명 7

최종 미션은 학생이 자기 노트북에서 실행 결과와 저장 파일을 함께 제출하는 형태로 운영한다. 강사는 결과 파일 이름, 행 수, 요약 문장 세 가지를 우선 확인하면 빠르게 피드백할 수 있다.
